In [3]:
import pandas as pd
import sqlite3

csv_url = (
    "https://raw.githubusercontent.com/"
    "JediNoahMio65/"
    "wearable-heart-rate-manufacturing-quality-dashboard/"
    "main/data/synthetic_manufacturing_data.csv"
)

df = pd.read_csv(csv_url)

connection = sqlite3.connect(":memory:")

df.to_sql(
    "manufacturing_quality",
    connection,
    if_exists="replace",
    index=False
)

row_count = pd.read_sql_query(
    "SELECT COUNT(*) AS rows_loaded FROM manufacturing_quality;",
    connection
)

display(row_count)

,rows_loaded
0,2400


In [4]:
queries = {
    "Query 1: Overall manufacturing-quality KPIs": """
        SELECT
            COUNT(*) AS total_units_tested,
            SUM(CASE WHEN test_status = 'Pass' THEN 1 ELSE 0 END) AS units_passed,
            SUM(CASE WHEN test_status = 'Fail' THEN 1 ELSE 0 END) AS units_failed,
            ROUND(
                100.0 * SUM(CASE WHEN test_status = 'Pass' THEN 1 ELSE 0 END)
                / COUNT(*), 1
            ) AS first_pass_yield_pct,
            ROUND(
                100.0 * SUM(CASE WHEN test_status = 'Fail' THEN 1 ELSE 0 END)
                / COUNT(*), 1
            ) AS failure_rate_pct,
            ROUND(
                100.0 * SUM(CASE WHEN rework_required = 'Yes' THEN 1 ELSE 0 END)
                / COUNT(*), 1
            ) AS rework_rate_pct
        FROM manufacturing_quality
    """,

    "Query 2: First-pass yield by production line": """
        SELECT
            production_line,
            COUNT(*) AS units_tested,
            SUM(CASE WHEN test_status = 'Pass' THEN 1 ELSE 0 END) AS units_passed,
            ROUND(
                100.0 * AVG(CASE WHEN test_status = 'Pass' THEN 1.0 ELSE 0.0 END), 1
            ) AS first_pass_yield_pct,
            ROUND(AVG(heart_rate_error_bpm), 2) AS mean_heart_rate_error_bpm,
            ROUND(AVG(response_time_seconds), 2) AS mean_response_time_seconds,
            ROUND(AVG(signal_quality_score), 1) AS mean_signal_quality_score
        FROM manufacturing_quality
        GROUP BY production_line
        ORDER BY first_pass_yield_pct ASC
    """,

    "Query 3: First-pass yield by test station": """
        SELECT
            test_station,
            COUNT(*) AS units_tested,
            ROUND(
                100.0 * AVG(CASE WHEN test_status = 'Pass' THEN 1.0 ELSE 0.0 END), 1
            ) AS first_pass_yield_pct,
            ROUND(AVG(signal_quality_score), 1) AS mean_signal_quality_score,
            SUM(CASE WHEN defect_category = 'Low signal quality' THEN 1 ELSE 0 END)
                AS low_signal_quality_failures
        FROM manufacturing_quality
        GROUP BY test_station
        ORDER BY first_pass_yield_pct ASC
    """,

    "Query 4: Firmware comparison": """
        SELECT
            firmware_version,
            COUNT(*) AS units_tested,
            ROUND(
                100.0 * AVG(CASE WHEN test_status = 'Pass' THEN 1.0 ELSE 0.0 END), 1
            ) AS first_pass_yield_pct,
            ROUND(AVG(response_time_seconds), 2) AS mean_response_time_seconds,
            ROUND(AVG(heart_rate_error_bpm), 2) AS mean_heart_rate_error_bpm
        FROM manufacturing_quality
        GROUP BY firmware_version
        ORDER BY firmware_version
    """,

    "Query 5: Defect Pareto analysis with cumulative percentage": """
        WITH defect_counts AS (
            SELECT
                defect_category,
                COUNT(*) AS defect_count
            FROM manufacturing_quality
            WHERE defect_category <> 'None'
            GROUP BY defect_category
        ),
        ranked_defects AS (
            SELECT
                defect_category,
                defect_count,
                ROUND(
                    100.0 * defect_count / SUM(defect_count) OVER (), 1
                ) AS percent_of_defects,
                ROUND(
                    100.0 * SUM(defect_count) OVER (
                        ORDER BY defect_count DESC, defect_category
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                    ) / SUM(defect_count) OVER (), 1
                ) AS cumulative_percent
            FROM defect_counts
        )
        SELECT
            defect_category,
            defect_count,
            percent_of_defects,
            cumulative_percent
        FROM ranked_defects
        ORDER BY defect_count DESC, defect_category
    """,

    "Query 6: Daily yield trend, excluding low-volume days": """
        SELECT
            build_date,
            COUNT(*) AS units_tested,
            ROUND(
                100.0 * AVG(CASE WHEN test_status = 'Pass' THEN 1.0 ELSE 0.0 END), 1
            ) AS first_pass_yield_pct
        FROM manufacturing_quality
        GROUP BY build_date
        HAVING COUNT(*) >= 20
        ORDER BY build_date
    """,
}

for title, query in queries.items():
    print(f"\n{title}")
    display(pd.read_sql_query(query, connection))


Query 1: Overall manufacturing-quality KPIs


,total_units_tested,units_passed,units_failed,first_pass_yield_pct,failure_rate_pct,rework_rate_pct
0,2400,1906,494,79.4,20.6,15.0



Query 2: First-pass yield by production line


,production_line,units_tested,units_passed,first_pass_yield_pct,mean_heart_rate_error_bpm,mean_response_time_seconds,mean_signal_quality_score
0,Line B,810,615,75.9,2.96,4.09,90.4
1,Line C,575,454,79.0,2.46,4.10,90.2
2,Line A,1015,837,82.5,2.43,4.08,90.8



Query 3: First-pass yield by test station


...

,firmware_version,units_tested,first_pass_yield_pct,mean_response_time_seconds,mean_heart_rate_error_bpm
0,FW-SIM-0.2,1244,75.1,4.32,2.63
1,FW-SIM-0.3,1156,84.1,3.84,2.60



Query 5: Defect Pareto analysis with cumulative percentage


,defect_category,defect_count,percent_of_defects,cumulative_percent
0,Slow response time,205,41.5,41.5
1,Low signal quality,114,23.1,64.6
2,High heart-rate error,100,20.2,84.8
3,Assembly defect,47,9.5,94.3
4,Cosmetic defect,28,5.7,100.0



Query 6: Daily yield trend, excluding low-volume days


,build_date,units_tested,first_pass_yield_pct
0,2026-07-01,89,77.5
1,2026-07-02,89,75.3
2,2026-07-03,89,78.7
3,2026-07-04,89,73.0
4,2026-07-05,89,78.7
5,2026-07-06,89,77.5
6,2026-07-07,88,71.6
7,2026-07-08,89,77.5
8,2026-07-09,89,83.1
9,2026-07-10,89,68.5


In [5]:
sql_file_contents = """-- ============================================================
-- Simulated Wearable Manufacturing Quality SQL Analysis
-- Source: synthetic_manufacturing_data.csv
-- Educational portfolio project. Fictional data only.
-- ============================================================

-- Query 1: Overall manufacturing-quality KPIs
SELECT
    COUNT(*) AS total_units_tested,
    SUM(CASE WHEN test_status = 'Pass' THEN 1 ELSE 0 END) AS units_passed,
    SUM(CASE WHEN test_status = 'Fail' THEN 1 ELSE 0 END) AS units_failed,
    ROUND(
        100.0 * SUM(CASE WHEN test_status = 'Pass' THEN 1 ELSE 0 END)
        / COUNT(*),
        1
    ) AS first_pass_yield_pct,
    ROUND(
        100.0 * SUM(CASE WHEN test_status = 'Fail' THEN 1 ELSE 0 END)
        / COUNT(*),
        1
    ) AS failure_rate_pct,
    ROUND(
        100.0 * SUM(CASE WHEN rework_required = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        1
    ) AS rework_rate_pct
FROM manufacturing_quality;

-- Query 2: First-pass yield by production line
SELECT
    production_line,
    COUNT(*) AS units_tested,
    SUM(CASE WHEN test_status = 'Pass' THEN 1 ELSE 0 END) AS units_passed,
    ROUND(
        100.0 * AVG(CASE WHEN test_status = 'Pass' THEN 1.0 ELSE 0.0 END),
        1
    ) AS first_pass_yield_pct,
    ROUND(AVG(heart_rate_error_bpm), 2) AS mean_heart_rate_error_bpm,
    ROUND(AVG(response_time_seconds), 2) AS mean_response_time_seconds,
    ROUND(AVG(signal_quality_score), 1) AS mean_signal_quality_score
FROM manufacturing_quality
GROUP BY production_line
ORDER BY first_pass_yield_pct ASC;

-- Query 3: First-pass yield by test station
SELECT
    test_station,
    COUNT(*) AS units_tested,
    ROUND(
        100.0 * AVG(CASE WHEN test_status = 'Pass' THEN 1.0 ELSE 0.0 END),
        1
    ) AS first_pass_yield_pct,
    ROUND(AVG(signal_quality_score), 1) AS mean_signal_quality_score,
    SUM(CASE WHEN defect_category = 'Low signal quality' THEN 1 ELSE 0 END)
        AS low_signal_quality_failures
FROM manufacturing_quality
GROUP BY test_station
ORDER BY first_pass_yield_pct ASC;

-- Query 4: Firmware comparison
SELECT
    firmware_version,
    COUNT(*) AS units_tested,
    ROUND(
        100.0 * AVG(CASE WHEN test_status = 'Pass' THEN 1.0 ELSE 0.0 END),
        1
    ) AS first_pass_yield_pct,
    ROUND(AVG(response_time_seconds), 2) AS mean_response_time_seconds,
    ROUND(AVG(heart_rate_error_bpm), 2) AS mean_heart_rate_error_bpm
FROM manufacturing_quality
GROUP BY firmware_version
ORDER BY firmware_version;

-- Query 5: Defect Pareto analysis with cumulative percentage
WITH defect_counts AS (
    SELECT
        defect_category,
        COUNT(*) AS defect_count
    FROM manufacturing_quality
    WHERE defect_category <> 'None'
    GROUP BY defect_category
),
ranked_defects AS (
    SELECT
        defect_category,
        defect_count,
        ROUND(
            100.0 * defect_count / SUM(defect_count) OVER (),
            1
        ) AS percent_of_defects,
        ROUND(
            100.0 * SUM(defect_count) OVER (
                ORDER BY defect_count DESC, defect_category
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) / SUM(defect_count) OVER (),
            1
        ) AS cumulative_percent
    FROM defect_counts
)
SELECT
    defect_category,
    defect_count,
    percent_of_defects,
    cumulative_percent
FROM ranked_defects
ORDER BY defect_count DESC, defect_category;

-- Query 6: Daily yield trend, excluding low-volume days
SELECT
    build_date,
    COUNT(*) AS units_tested,
    ROUND(
        100.0 * AVG(CASE WHEN test_status = 'Pass' THEN 1.0 ELSE 0.0 END),
        1
    ) AS first_pass_yield_pct
FROM manufacturing_quality
GROUP BY build_date
HAVING COUNT(*) >= 20
ORDER BY build_date;
"""

with open("01_manufacturing_quality_analysis.sql", "w") as file:
    file.write(sql_file_contents)

print("Created 01_manufacturing_quality_analysis.sql")

Created 01_manufacturing_quality_analysis.sql
